Plot line counts for a single repository over time. Collect the data first with:

    code-metrics repo-history https://github.com/lsst/daf_butler

In [ ]:
%matplotlib widget

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from lsst.codemetrics.plotting import (
    CPP_HEADER_ALIASES,
    apply_aliases,
    load_repo,
    pivot,
    select,
    top_series,
)

In [ ]:
name = "daf_butler"
frame = load_repo(name)
frame.head()

In [ ]:
# select() with no counter= uses the file's sole backend and raises if
# it holds several. Pass counter="cloc" to choose one explicitly.
chosen = select(frame)

# Fold C and C++ headers into C++ so they plot as one series.
# Covers all three backends; unused keys are simply no-ops.
folded = apply_aliases(chosen, CPP_HEADER_ALIASES)

# Any real repository reports enough languages to bury the plot under
# its own legend. Rank language-and-measure pairs by the largest each
# ever reached, so a subsystem that grew and was later removed still
# shows its rise and fall, and a series that is flat zero (JSON has no
# comments) never takes a slot from one that carries content.
series = top_series(folded, n=10)
series

In [ ]:
wide = {measure: pivot(folded, value=measure) for _, measure in series}

fig, ax = plt.subplots()
for language, measure in series:
    line = wide[measure][language]
    ax.plot(line.index, line, label=f"{language} {measure}", drawstyle="steps-post")
ax.set_title(f"Lines of {name} code and comments")
ax.set_ylim(bottom=0)

# Let the tick labels follow the span. A repository covering a few
# months would otherwise get nine "2025-12"-style labels that collide,
# where one covering years gets bare years that fit comfortably.
locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

# Park the legend outside the axes so it can never cover the curves.
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0, fontsize="small")

In [ ]:
# Uncomment to save a PDF for publications.
# ax.set_title("")
# fig.savefig(f"{name}-lines.pdf", bbox_inches="tight")